# Book recommendation System

- Web scraping: Popular lists from Goodreads, link: https://www.goodreads.com/list/popular_lists
- API retrieval: OpenLibrary, link: https://openlibrary.org/developers/api
* information on API usage find here: https://openlibrary.org/dev/docs/api/search

Information to extract from each source:

Book name, author, raiting in stars, language, published date, pages, ISBN 10, genres/subjects (goodreads/openlibrary),book cover image

## 1. Web scraping: Goodreads

In [2]:
# 📚 Basic libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

url='https://www.goodreads.com/list/popular_lists'
# Now we identify ourserlves as "humans", and not bots to avoid being blocked.
headers = {
  "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}
response = requests.get(url, headers=headers)
print(response) # we check the response is correct =200
print(response.headers['Content-Type']) # we check our content is html
soup_lists = BeautifulSoup(response.content, 'html.parser') #we store the html info in soup with a nice structure
type(soup_lists)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
<Response [200]>
text/html; charset=utf-8


bs4.BeautifulSoup

In [3]:
# ⚙️ Settings
pd.set_option('display.max_columns', None) # display all columns
import warnings
warnings.filterwarnings('ignore') # ignore warnings

In [4]:
#Now is time to start finding the tags, attributes we need
#we look for the tags that includes inforation from the book lists displayed on the main page
list_tags= soup_lists.find_all('a',class_='listTitle')
print(len(list_tags))
print(list_tags[0])

10
<a class="listTitle" href="/list/show/1.Best_Books_Ever">Best Books Ever</a>


In [5]:
#Now extracting data needed only to loop in it later
from urllib.parse import urljoin

BASE = "https://www.goodreads.com"

lists_data = []
for tag in list_tags:
    lists_data.append({
        "list_name": tag.get_text(strip=True),
        "list_url": urljoin(BASE, tag["href"])   # relative → absolute
    })

print(lists_data[0])
# {'list_name': 'Best Books Ever', 'list_url': 'https://www.goodreads.com/list/show/1.Best_Books_Ever'}

{'list_name': 'Best Books Ever', 'list_url': 'https://www.goodreads.com/list/show/1.Best_Books_Ever'}


In [6]:
#function to go to an specific link
def go_url(link):
    # Now we identify ourserlves as "humans", and not bots to avoid being blocked.
    headers = {
      "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
    }
    response = requests.get(link, headers=headers)
    print(f"The response code is: {response}") # we check the response is correct =200
    soupy = BeautifulSoup(response.content, 'html.parser') #we store the html info in soup with a nice structure
    return soupy

In [7]:
link=lists_data[8]['list_url']
soup_books= go_url(link)

The response code is: <Response [200]>


In [27]:
book_tags= soup_books.find_all('a',class_='bookTitle')
print(len(book_tags))

ls_books = []
for book in book_tags:
    ls_books.append({
        "book_name": book.find('span', itemprop='name').getText().split('(')[0].strip(' '),
        "book_url": urljoin(BASE, book["href"])   # relative → absolute
    })

print(ls_books[0])

{'book_name': 'The Hunger Games', 'book_url': 'https://www.goodreads.com/book/show/2767052-the-hunger-games'}


In [79]:
link_b=ls_books[0]['book_url']
soup_b= go_url(link_b)

The response code is: <Response [200]>


In [80]:
#Now in the book page, extract all information about the book
book_data= soup_b.find('div',class_='BookPageMetadataSection')
author = soup_b.select_one('span.ContributorLink__name').get_text(strip=True)
rating = float(soup_b.select_one('div.RatingStatistics__rating').get_text(strip=True))
synopsis= soup_b.select_one('span.Formatted').get_text(strip=True)

import re

#Extract Pages
pages_text = soup_b.select_one('[data-testid="pagesFormat"]').get_text(strip=True) # "374 pages, Hardcover"
match = re.search(r'(\d[\d,]*)\s*pages', pages_text)
pages = int(match.group(1).replace(',', ''))

from datetime import datetime

pub_text = soup_b.select_one('[data-testid="publicationInfo"]').get_text(strip=True) # "First published September 14, 2008"

#Clean up the prefix string cleanly (Result: "September 14, 2008")
pub_date_str = pub_text.replace("First published", "").strip()
#CONVERT the string into a datetime object (The missing step!)
date_obj = datetime.strptime(pub_date_str, "%B %d, %Y")
#Format the datetime object to standard YYYY-MM-DD string
pub_date_stnd = date_obj.strftime("%Y-%m-%d")

genre_tags = soup_b.select("div[data-testid='genresList'] span.Button__labelItem")
genres = [g.get_text(strip=True) for g in genre_tags]
genres = [g for g in genres if g != "...more"]

#Book cover image
img_el = soup_b.find('img', class_='ResponsiveImage')
img_url = img_el.get('src')


In [8]:
# ============================================================
# Week 9 — Book Recommendation System
# Data Sourcing: Web Scraping (Goodreads Listopia + Book Detail Pages)
# ============================================================

import os
import re
import time
import random
from datetime import datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

%load_ext autoreload
%autoreload 2
from functions import read_file, out_csv

# ---------- CONFIG ----------
BASE = "https://www.goodreads.com"
HEADERS = {
    "user-agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}
YAML_PATH = "../config.yaml"  # point 3: hoisted once, no longer recomputed in loop


def go_url(url, headers=HEADERS, timeout=10):
    """Fetch URL, return BeautifulSoup or None on failure. Never raises — caller checks for None."""
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code != 200:
            print(f"⚠️ status {resp.status_code} — {url}")
            return None
        return BeautifulSoup(resp.content, 'html.parser')
    except requests.RequestException as e:
        print(f"⚠️ request failed — {url} — {e}")
        return None


# ============================================================
# STAGE 1 — list-of-lists (Listopia popular lists)
# ============================================================
soup_lists = go_url(f"{BASE}/list/popular_lists")
list_tags = soup_lists.find_all('a', class_='listTitle')

lists_data = [
    {"list_name": tag.get_text(strip=True), "list_url": urljoin(BASE, tag["href"])}
    for tag in list_tags
]
print(f"Lists found: {len(lists_data)}")


# ============================================================
# STAGE 2 — books per list, deduped across all lists
# ============================================================
seen_urls = set()
ls_books = []

for lst in lists_data:
    soup_books = go_url(lst["list_url"])
    if soup_books is None:
        continue

    for book in soup_books.find_all('a', class_='bookTitle'):
        name_el = book.find('span', itemprop='name')
        if name_el is None:
            continue
        book_url = urljoin(BASE, book["href"])
        if book_url in seen_urls:
            continue
        seen_urls.add(book_url)
        ls_books.append({
            "book_name": name_el.get_text(strip=True).split('(')[0].strip(),
            "book_url": book_url,
            "source_list": lst["list_name"],
        })
    time.sleep(random.uniform(3, 6))

print(f"Unique books collected: {len(ls_books)}")
assert len(ls_books) == len(set(b['book_url'] for b in ls_books)), "Duplicate URLs slipped through dedup"


# ============================================================
# STAGE 3 — detail scrape, resumable, None-safe, incremental save
# ============================================================

# point 5: resume — load already-scraped rows if a prior run exists
try:
    existing_df = read_file(YAML_PATH, 'raw_data', 'raw_scraped')
    done_urls = set(existing_df["book_url"])
    records = existing_df.to_dict("records")
except Exception:
    done_urls = set()
    records = []

ls_books_remaining = [b for b in ls_books if b["book_url"] not in done_urls]
print(f"Already scraped: {len(done_urls)} — remaining: {len(ls_books_remaining)}")

for i, book_i in enumerate(ls_books_remaining):
    soup_b = go_url(book_i["book_url"])
    if soup_b is None:
        continue

    author_el = soup_b.select_one('span.ContributorLink__name')
    author = author_el.get_text(strip=True) if author_el else None

    # point 2: rating guard — element present ≠ text parseable ("New", empty, etc.)
    rating_el = soup_b.select_one('div.RatingStatistics__rating')
    rating = None
    if rating_el:
        try:
            rating = float(rating_el.get_text(strip=True))
        except ValueError:
            rating = None

    synopsis_el = soup_b.select_one('span.Formatted')
    synopsis = synopsis_el.get_text(strip=True) if synopsis_el else None

    pages_el = soup_b.select_one('[data-testid="pagesFormat"]')
    pages_text = pages_el.get_text(strip=True) if pages_el else None
    match = re.search(r'(\d[\d,]*)\s*pages', pages_text) if pages_text else None
    pages = int(match.group(1).replace(',', '')) if match else None

    pub_el = soup_b.select_one('[data-testid="publicationInfo"]')
    pub_date_std = None
    if pub_el:
        pub_text = pub_el.get_text(strip=True).replace("First published", "").strip()
        try:
            pub_date_std = datetime.strptime(pub_text, "%B %d, %Y").strftime("%Y-%m-%d")
        except ValueError:
            pub_date_std = None  # e.g. year-only or non-standard format

    genre_tags = soup_b.select("div[data-testid='genresList'] span.Button__labelItem")
    genres = [g.get_text(strip=True) for g in genre_tags if g.get_text(strip=True) != "...more"]
    genres_str = "|".join(genres) if genres else None

    img_el = soup_b.find('img', class_='ResponsiveImage')
    img_url = img_el.get('src') if img_el else None

    records.append({
        "book_name": book_i["book_name"],
        "book_url": book_i["book_url"],
        "source_list": book_i["source_list"],
        "author": author,
        "rating": rating,
        "synopsis": synopsis,
        "pages": pages,
        "pub_date": pub_date_std,
        "genres": genres_str,
        "img_url": img_url,
    })

    if (i + 1) % 25 == 0:
        scraped_df = pd.DataFrame(records)
        out_csv(scraped_df, YAML_PATH, 'raw_data', 'raw_scraped')  # point 3: YAML_PATH referenced, not recomputed
        print(f"Checkpoint saved — {i + 1} new this run, {len(records)} total")

    time.sleep(random.uniform(1, 2))

# final save
scraped_df = pd.DataFrame(records)
out_csv(scraped_df, YAML_PATH, 'raw_data', 'raw_scraped')
print(f"Done — {len(records)} total books saved")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Lists found: 10
Unique books collected: 701
Already scraped: 0 — remaining: 701
⚠️ status 503 — https://www.goodreads.com/book/show/11870085-the-fault-in-our-stars
⚠️ status 503 — https://www.goodreads.com/book/show/28187.The_Lightning_Thief
File saved to: ../data/raw/goodreads.csv
Checkpoint saved — 25 new this run, 23 total
File saved to: ../data/raw/goodreads.csv
Checkpoint saved — 50 new this run, 48 total
File saved to: ../data/raw/goodreads.csv
Checkpoint saved — 75 new this run, 73 total
⚠️ status 503 — https://www.goodreads.com/book/show/9293020-the-elephant-tree
⚠️ status 503 — https://www.goodreads.com/book/show/345627.Vampire_Academy
File saved to: ../data/raw/goodreads.csv
Checkpoint saved — 100 new this run, 96 total
⚠️ status 503 — https://www.goodreads.com/book/show/1420.Hamlet
⚠️ status 503 — https://www.goodreads.com/book/show/5886881-dark-places
⚠️ status 503 — https://www.goodread